<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Bibliotecas" data-toc-modified-id="Bibliotecas-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Bibliotecas</a></span></li><li><span><a href="#Pegando-dados-do-Postgres" data-toc-modified-id="Pegando-dados-do-Postgres-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Pegando dados do Postgres</a></span></li><li><span><a href="#Funções" data-toc-modified-id="Funções-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Funções</a></span></li><li><span><a href="#Calcular-o-Information-Value-e-WOE" data-toc-modified-id="Calcular-o-Information-Value-e-WOE-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Calcular o Information Value e WOE</a></span></li><li><span><a href="#Matriz-de-correlação" data-toc-modified-id="Matriz-de-correlação-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Matriz de correlação</a></span></li><li><span><a href="#Modelagem:-Modelo-Logístico" data-toc-modified-id="Modelagem:-Modelo-Logístico-6"><span class="toc-item-num">6&nbsp;&nbsp;</span>Modelagem: Modelo Logístico</a></span></li><li><span><a href="#Avaliação" data-toc-modified-id="Avaliação-7"><span class="toc-item-num">7&nbsp;&nbsp;</span>Avaliação</a></span></li></ul></div>

# Bibliotecas

In [1]:
import psycopg2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from unidecode import unidecode 

# Pegando dados do Postgres

In [3]:
sql = '''
SELECT *
FROM vw_ocorrencia_deslizamento_chuva
'''

database = ''
user = ''
password = ''
host = ''
port = ''

conn = psycopg2.connect(database=database, user=user, password=password, host=host, port=port)

with conn.cursor() as cursor:

    cursor.execute(sql)

    ocorrencia_deslizamento_chuva = cursor.fetchall()

    cols = []
    [cols.append(nm[0]) for nm in cursor.description]

    df = pd.DataFrame(ocorrencia_deslizamento_chuva)
    df.columns = cols

#trata o nome da estacao que pode dar erro por causa de acento e espaços
nome_estacao = df['nome_estacao'].to_list()
new_nome_estacao = [unidecode(nome).replace(" ", "_").replace("'","") for nome in nome_estacao]
df['nome_estacao']= df['nome_estacao'].replace(nome_estacao, new_nome_estacao)

df.head(5)

,indice_pluv,quinzemin,trintamin,umahora,seishoras,dozehoras,vintequatrohoras,quarentaoitohoras,setentaduashoras,noventaseishoras,...,id_estacao,nome_estacao,lat_est,long_est,ocorrencia,ameaca_prevencao,mapeamento,qtd_solicitacoes,latitude,long
0,0.0,0.0,0.0,0.0,0.2,52.40933449273803683,53.58737449273803703,53.58737449273803703,53.58737449273803703,53.58737449273803703,...,186,CHARITAS,-22.9340000000000,-43.0980000000000,1,0,0,1,-22.9362756210453,-43.0975550015999
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,184,JURUJUBA,-22.9320000000000,-43.1170000000000,1,0,0,1,-22.9301070941738,-43.1179193533920
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.4,...,189,FONSECA,-22.8830000000000,-43.0830000000000,1,0,0,1,-22.8851855550989,-43.0886878554080
3,0.00,0.20,0.20,0.20,23.27,23.27,23.27,24.07,24.07,24.07,...,176,LARGO_DA_BATALHA,-22.9070000000000,-43.0670000000000,1,0,0,1,-22.9108171115432,-43.0540950758103
4,0.00,0.00,0.20,0.40,1.20,24.67,24.67,25.47,25.47,25.47,...,176,LARGO_DA_BATALHA,-22.9070000000000,-43.0670000000000,1,0,0,1,-22.8992117471139,-43.0679938402780


# Funções

In [ ]:
##################
#### WoE e IV ####
##################

def WoE_IV(variavel, rotulo, dados)
df_woe_iv = (pd.crosstab(dados[variavel],dados[rotulo],
                      normalize='columns')
             .assign(woe=lambda dfx: np.log(dfx[1] / dfx[0]))
             .assign(iv=lambda dfx: np.sum(dfx['woe']*
                                           (dfx[1]-dfx[0]))))

print(df_woe_iv)

In [ ]:
#################################
###### Top ABS Correlations #####
#################################


def get_redundant_pairs(df):
    '''Get diagonal and lower triangular pairs of correlation matrix'''
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i+1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop

def get_top_abs_correlations(df, n=5):
    au_corr = df.corr().abs().unstack()
    labels_to_drop = get_redundant_pairs(df)
    au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
    return au_corr[0:n]

In [ ]:
##############################
###### Matriz de confusão ####
##############################

def matriz_confusao(y_real,y_predito,modelo):

### Grafico ###

    tabela=confusion_matrix(y_real,y_predito)

    group_names = ["True Neg","False Pos","False Neg","True Pos"]
    group_counts = ["{0:0.0f}".format(value) for value in
                tabela.flatten()]
    group_percentages = ["{0:.5%}".format(value) for value in
                     tabela.flatten()/np.sum(tabela)]
    labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
          zip(group_names,group_counts,group_percentages)]
    labels = np.asarray(labels).reshape(2,2)
    f = plt.figure()
    f.set_figwidth(8)
    f.set_figheight(8)

    sns.heatmap(tabela, annot=labels, fmt="", cmap='Blues')

### Tabela ###
    Resultados=PrettyTable()
    Resultados.field_names=["Métrica","Resultado"]
    Resultados.title= modelo
    Resultados.align["Métrica"]="l"
    Resultados.align["Resultado"]="r"

    Resultados.add_row(["Acurácia:",round(sklearn.metrics.accuracy_score(y_real,y_predito),2)])
    Resultados.add_row(["Precisão:",round(sklearn.metrics.precision_score(y_real,y_predito),2)])
    Resultados.add_row(["Recall:",round(sklearn.metrics.recall_score(y_real,y_predito),2)])
    Resultados.add_row(["F1-Score:",round(sklearn.metrics.f1_score(y_real,y_predito),2)])

    print(Resultados)
  
    return

# Calcular o Information Value e WOE

Objetivo: Agrupamento

Como fazer? 
- Analisar a semelhança quanto a discriminação da variável resposta.
- Avaliar o nº de casos em cada atributo de forma que esta quantidade seja representativa.
- Agrupar de forma que faça sentido.

Benefícios:
- Preparar as variáveis para o modelo.
- A equação do modelo fica mais simples.
- Confere estabilidade ao modelo, minimizando o risco de overfitting

O valor da informação sempre cai quando as categorias da variável são agrupadas. Sempre devemos combinar categorias com WoE similares.

Classificação do IV quanto a preditividade:

< 0,02 -> Não útil çpara predição

0,02 - 0,1 -> Poder preditivo fraco

0,1 - 0,3 -> Poder preditivo médio

0,3 - 0,5 -> Poder preditivo forte

\> 0,5 -> Poder preditivo suspeito

# Matriz de correlação

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df.corr(),annot=True)

In [ ]:
print(get_top_abs_correlations(df, 10))

# Modelagem: Modelo Logístico

Obs: Stepwise tem apenas no R

In [ ]:
modelo5 = smf.glm(formula='Survived ~ FLG_Sex + FLG_Cabin_Letra_BCDE + Pclass +  Age_Agrup_35_80 -1 ', data=train,
                family = sm.families.Binomial()).fit()

print(modelo5.summary())
print(modelo5.aic)

# Avaliação